# **Group 7: Future Arctic sea ice change**

**Students :**
- Julie Descloitres
- Adam Lagssaibi
- Ewen Cottour
- Edgard Dabier

In [ ]:
!pip install zarr==2.14.2
!pip install cftime
!pip install nc-time-axis
!pip install gcsfs
!pip uninstall shapely --yes
!pip install lida==0.0.10 shapely cartopy --no-binary shapely --no-binary cartopy --use-deprecated=legacy-resolver
!wget https://raw.githubusercontent.com/SciTools/cartopy/main/lib/cartopy/feature/download/__main__.py -O cartopy_feature_download.py
!python cartopy_feature_download.py physical

In [ ]:
from matplotlib import pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr
import zarr
import gcsfs

import cartopy.crs as ccrs
from matplotlib.axes import Axes
from cartopy.mpl.geoaxes import GeoAxes
GeoAxes._pcolormesh_patched = Axes.pcolormesh

xr.set_options(display_style='html')
%matplotlib inline

### Loading the datasets

In [ ]:
df = pd.read_csv('https://storage.googleapis.com/cmip6/cmip6-zarr-consolidated-stores.csv')

In [ ]:
activity_ID   = 'CMIP'
source_ID     = 'CMCC-ESM2'
experiment_ID = 'piControl'
variable_ID   = 'siconc'

In [ ]:
experiment_ID_dic = {'Julie':'1pctCO2',
                     'Adam':'abrupt-4xCO2',
                     'Ewen':'historical',
                     'Edgard':'piControl'}

# Senario: piControl

In [ ]:
experiment_ID=experiment_ID_dic['Julie']

In [ ]:
df_ssh = df.query(f"activity_id == '{activity_ID}' & source_id == '{source_ID}' & experiment_id == '{experiment_ID}' & variable_id == '{variable_ID}'")
df_ssh

In [ ]:
gcs = gcsfs.GCSFileSystem(token='anon')
zstore = df_ssh.zstore.values[-1]
mapper = gcs.get_mapper(zstore)
ds = xr.open_zarr(mapper, consolidated=True,decode_times=True)

Observation of the name given to the variables in the dataset :

In [ ]:
ds.coords

#### Now, we'll focus only on the arctic area (>70° N) :

In [ ]:
lat = ds.latitude.compute()
ds=ds.where(lat > 70,drop=True)
ds

We can plot the values of sinconc (the concentration of sea ice) onto a map :

In [ ]:
ax = plt.axes(projection=ccrs.NorthPolarStereo());
ax.set_extent([-180, 180, 70, 90], ccrs.PlateCarree())
ds.siconc[0].plot.pcolormesh(ax=ax,transform=ccrs.PlateCarree(),x='longitude', y='latitude', add_colorbar=True);
ax.coastlines();
ax.gridlines(draw_labels=True, x_inline=False, y_inline=True,color='grey');

Now that we have the dataset, we want to change its format so that we have one column for every season (winter, spring, summer and autumn) for every year (between 1998 and 2015) for every couple (lon, lat)

## Dataset Preparation


In [ ]:
ds = ds.resample(time='QS-DEC').mean(dim="time")
# ds.where(ds.time.dt.season == 'DJF', drop = True).isel(i = 10, j = 150).siconc.plot()
ds['siconc'].where(ds.time.dt.season == 'DJF', drop = True)

Seasonal dataset

In [ ]:
seasons = {
    'winter': 'DJF',
    'spring': 'MAM',
    'summer': 'JJA',
    'autumn': 'SON'
}

seasonal_data = {
    season: ds.where(ds.time.dt.season == seasons[season], drop = True).siconc
    for season in seasons
}

In [ ]:
ds_seasonal = xr.concat([seasonal_data[season] for season in seasons], dim='season')
ds_seasonal['season'] = list(seasons.keys())

# ds_seasonal

In [ ]:
df_winter = seasonal_data['winter'].to_dataframe().reset_index()
df_spring = seasonal_data['spring'].to_dataframe().reset_index()
df_summer = seasonal_data['summer'].to_dataframe().reset_index()
df_autumn = seasonal_data['autumn'].to_dataframe().reset_index()

In [ ]:
# Pivot table for desired format
df_pivot_winter = df_winter.pivot_table(
    index=['longitude', 'latitude'],
    columns=['time'],
    values='siconc'
)
df_pivot_winter.columns = [f"winter_{col.year}" for col in df_pivot_winter.columns]
df_pivot_winter.reset_index(inplace=True)

df_pivot_spring = df_spring.pivot_table(
    index=['longitude', 'latitude'],
    columns=['time'],
    values='siconc'
)
df_pivot_spring.columns = [f"spring_{col.year}" for col in df_pivot_spring.columns]
df_pivot_spring.reset_index(inplace=True)

df_pivot_summer = df_summer.pivot_table(
    index=['longitude', 'latitude'],
    columns=['time'],
    values='siconc'
)
df_pivot_summer.columns = [f"summer_{col.year}" for col in df_pivot_summer.columns]
df_pivot_summer.reset_index(inplace=True)

df_pivot_autumn = df_autumn.pivot_table(
    index=['longitude', 'latitude'],
    columns=['time'],
    values='siconc'
)
df_pivot_autumn.columns = [f"autumn_{col.year}" for col in df_pivot_autumn.columns]
df_pivot_autumn.reset_index(inplace=True)

## Clustering

#### Prepare for clustering by season:

In [ ]:
# df_pivot = df_pivot_autumn
df_pivot = df_pivot_winter
# df_pivot = df_pivot_spring
# df_pivot = df_pivot_summer

#### Prepare for clustering by year (mean of every season per year)

In [ ]:
# Merge the dataframes
df_pivot = pd.concat([df_pivot_winter,
                      df_pivot_spring.drop(columns=['longitude', 'latitude']),
                      df_pivot_summer.drop(columns=['longitude', 'latitude']),
                      df_pivot_autumn.drop(columns=['longitude', 'latitude'])], axis=1)

# Calculate the mean for each year
years = sorted(list(set([int(col.split('_')[1]) for col in df_pivot.columns if 'winter' in col])))
df_mean = pd.DataFrame()
df_mean['longitude'] = df_pivot['longitude']
df_mean['latitude'] = df_pivot['latitude']

for year in years:
    winter_col = f"winter_{year}"
    spring_col = f"spring_{year}"
    summer_col = f"summer_{year}"
    autumn_col = f"autumn_{year}"

    # Check if all columns exist before calculating the mean
    if all(col in df_pivot.columns for col in [winter_col, spring_col, summer_col, autumn_col]):
        df_mean[f"{year}"] = df_pivot[[winter_col, spring_col, summer_col, autumn_col]].mean(axis=1)
    else:
        print(f"Warning: Missing data for year {year}, skipping calculation")
        df_mean[f"{year}"] = np.nan  # or handle missing data as appropriate

df_pivot = df_mean

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

# Load dataset (assuming df_pivot is already prepared)
X = df_pivot.drop(columns=['longitude', 'latitude']).fillna(0)  # Remove spatial columns for clustering

# Normalize the data for better clustering
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Store the spatial coordinates separately for visualization
coords = df_pivot[['longitude', 'latitude']]

Seasonal datasets

In [ ]:
X_winter = df_pivot_winter.drop(columns=['longitude', 'latitude']).fillna(0)
X_spring = df_pivot_spring.drop(columns=['longitude', 'latitude']).fillna(0)
X_summer = df_pivot_summer.drop(columns=['longitude', 'latitude']).fillna(0)
X_autumn = df_pivot_autumn.drop(columns=['longitude', 'latitude']).fillna(0)

# Normalize the data for better clustering
scaler = StandardScaler()
X_winter_scaled = scaler.fit_transform(X_winter)
X_spring_scaled = scaler.fit_transform(X_spring)
X_summer_scaled = scaler.fit_transform(X_summer)
X_autumn_scaled = scaler.fit_transform(X_autumn)

# Store the spatial coordinates separately for visualization
coords = df_pivot[['longitude', 'latitude']]

Elbow Method (for K-Means)

In [ ]:
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from yellowbrick.cluster import KElbowVisualizer

# Use Elbow method to find the optimal number of clusters
model = KMeans(random_state=0)
visualizer = KElbowVisualizer(model, k=(2, 10), metric='distortion', timings=False)
visualizer.fit(X_scaled)
visualizer.show()

Silhouette Analysis

In [ ]:
from sklearn.metrics import silhouette_score

range_n_clusters = list(range(2, 10))
silhouette_scores = []

for n_clusters in range_n_clusters:
    kmeans = KMeans(n_clusters=n_clusters, random_state=0)
    cluster_labels = kmeans.fit_predict(X_scaled)
    silhouette_avg = silhouette_score(X_scaled, cluster_labels)
    silhouette_scores.append(silhouette_avg)

# Plot silhouette scores
plt.plot(range_n_clusters, silhouette_scores, marker='o')
plt.xlabel("Number of clusters")
plt.ylabel("Silhouette Score")
plt.title("Optimal Cluster Selection using Silhouette Score")
plt.show()

 k-Means Clustering

 We first tried several methods (K-Means, GMM, DBScan and Aglomerative) and selected the one with the highest silhouette score overall (K-Means)

 It turns out it is also the fastest and easiest method

In [ ]:
# Optimal number of clusters (from previous analysis)
optimal_clusters = 3

kmeans = KMeans(n_clusters=optimal_clusters, random_state=0)
df_pivot['kmeans_cluster'] = kmeans.fit_predict(X_scaled)

Seasonal clustering

In [ ]:
kmeans = KMeans(n_clusters=optimal_clusters, random_state=0)

df_pivot_winter['kmeans_cluster'] = kmeans.fit_predict(X_winter_scaled)
df_pivot_spring['kmeans_cluster'] = kmeans.fit_predict(X_spring_scaled)
df_pivot_summer['kmeans_cluster'] = kmeans.fit_predict(X_summer_scaled)
df_pivot_autumn['kmeans_cluster'] = kmeans.fit_predict(X_autumn_scaled)

Plotting the clusterings results on the map

In [ ]:
import cartopy.crs as ccrs
import cartopy.feature as cfeature

# Function to visualize clusters with legends
def plot_arctic_clusters_global(df_season, title):
    plt.figure(figsize=(12, 6))

    ax = plt.axes(projection=ccrs.NorthPolarStereo())
    ax.set_extent([-180, 180, 55, 90], crs=ccrs.PlateCarree())

    # Define a colormap with 3 fixed colors
    colors = ['midnightblue', 'royalblue', 'gold', 'orange', 'crimson']
    cmap = plt.cm.colors.ListedColormap(colors[:len(df_season['kmeans_cluster'].unique())])

    scatter = plt.scatter(
        df_season['longitude'], df_season['latitude'],
        c=df_season['kmeans_cluster'], cmap=cmap, s=10, edgecolor='k',
        transform=ccrs.PlateCarree()
    )

    ax.coastlines()
    ax.add_feature(cfeature.BORDERS, linewidth=0.5, edgecolor='gray')
    ax.add_feature(cfeature.LAND, facecolor='lightgray')
    ax.gridlines(draw_labels=True, linestyle="--", linewidth=0.5, color='gray')

    # Create legend with the same fixed colors
    legend_elements = []
    unique_clusters = df_season['kmeans_cluster'].unique()
    for i, cluster in enumerate(unique_clusters):
        legend_elements.append(plt.Line2D([0], [0], marker='o', label=f'Cluster {cluster}',
                                          markerfacecolor=colors[i % len(colors)],  # Use fixed colors
                                          markersize=8))

    ax.legend(handles=legend_elements, loc='upper right', bbox_to_anchor=(1.2, 1.0), fancybox=True, shadow=True)


    plt.title(title, fontsize=14)
    plt.xlabel('Longitude')
    plt.ylabel('Latitude')

    plt.show()

# Plot clustering results
plot_arctic_clusters_global(df_pivot_winter, 'K-Means Clustering winter')
plot_arctic_clusters_global(df_pivot_spring, 'K-Means Clustering spring')
plot_arctic_clusters_global(df_pivot_summer, 'K-Means Clustering summer')
plot_arctic_clusters_global(df_pivot_autumn, 'K-Means Clustering autumn')

In [ ]:
from sklearn.metrics import silhouette_score

def evaluate_clustering(df, column_name):
    labels = df[column_name]
    if len(set(labels)) > 1:  # Ensure there is more than one cluster
        silhouette = silhouette_score(X_scaled, labels)
        return silhouette
    else:
        return None

# Evaluate silhouette score for each clustering method
results = {
    'KMeans': evaluate_clustering(df_pivot, 'kmeans_cluster'),
    'Agglomerative': evaluate_clustering(df_pivot, 'agglo_cluster'),
    'DBSCAN': evaluate_clustering(df_pivot, 'dbscan_cluster'),
    'GMM': evaluate_clustering(df_pivot, 'gmm_cluster'),
}

# Print only silhouette scores
for method, score in results.items():
    if score is not None:
        print(f"{method} - Silhouette Score: {score:.3f}")
    else:
        print(f"{method} - Silhouette Score: Not applicable (single cluster or empty)")

#### Display the average evolution of each clusters

In [ ]:
method = 'kmeans_cluster'

df_pivot['cluster'] = df_pivot[f"{method}"]

# Select only numeric columns related to sea ice concentration changes over time
df_analysis = df_pivot.drop(columns=['longitude', 'latitude'])
df_analysis = df_analysis.loc[:, ~df_analysis.columns.str.contains('_cluster')]

# Calculate mean rate of change over time for each cluster
means_by_cluster = df_analysis.groupby('cluster').apply(
    lambda x: x.drop(columns=['cluster']).mean()
)

In [ ]:
means_by_cluster.columns = [col.replace('summer_', '') if 'summer_' in col else col for col in means_by_cluster.columns]
season = means_by_cluster.T

plt.plot(season.index, season[0], c='midnightblue', label='cluster 1')
plt.plot(season.index, season[1], c='royalblue', label='cluster 2')
plt.plot(season.index, season[2], c='gold', label='cluster 3')
plt.legend()
plt.ylabel('Percentage of sea ice extent')
plt.title("Evolution of the sea ice coverage for every clusters in summer")

# Remove vertical gridlines or reduce their frequency
plt.grid(axis='x', linestyle='--', linewidth=0.5, color='gray', alpha=0.7) #Example: keep only x gridlines

# Set x-axis ticks to display only 10 ticks
plt.xticks(season.index[::len(season.index)//10]) # Adjust the step as needed

plt.show()